In [39]:
using Pkg
Pkg.add("DecisionTree")
Pkg.add("CSV")
Pkg.add("DataFrames")

using DecisionTree
using CSV
using DataFrames

   Resolving package versions...
     Project No packages added to or removed from `C:\Users\tunak\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\tunak\.julia\environments\v1.12\Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `C:\Users\tunak\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\tunak\.julia\environments\v1.12\Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `C:\Users\tunak\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\tunak\.julia\environments\v1.12\Manifest.toml`


In [45]:
SHLD = CSV.read("C:\\Users\\tunak\\OneDrive\\Masaüstü\\Sleep_health_and_lifestyle_dataset.csv" , DataFrame)

Row,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps,Sleep Disorder
,Int64,String7,Int64,String31,Float64,Int64,Int64,Int64,String15,String7,Int64,Int64,String15
1,1,Male,27,Software Engineer,6.1,6,42,6,Overweight,126/83,77,4200,None
2,2,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,None
3,3,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,None
4,4,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea
5,5,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea
6,6,Male,28,Software Engineer,5.9,4,30,8,Obese,140/90,85,3000,Insomnia
7,7,Male,29,Teacher,6.3,6,40,7,Obese,140/90,82,3500,Insomnia
8,8,Male,29,Doctor,7.8,7,75,6,Normal,120/80,70,8000,None
9,9,Male,29,Doctor,7.8,7,75,6,Normal,120/80,70,8000,None


In [46]:
using DataFrames


function split_bp(bp_string)
    parts = split(bp_string, "/")
    return parse(Int, parts[1]), parse(Int, parts[2])
end


transform!(SHLD, "Blood Pressure" => ByRow(x -> split_bp(x)[1]) => :BP_Systolic)
transform!(SHLD, "Blood Pressure" => ByRow(x -> split_bp(x)[2]) => :BP_Diastolic)


select!(SHLD, Not("Blood Pressure"))



using DataFrames


bmi_mapping = Dict(
    "Normal" => 1,
    "Normal Weight" => 1, 
    "Overweight" => 2,
    "Obese" => 3
)


transform!(SHLD, "BMI Category" => ByRow(x -> bmi_mapping[x]) => "BMI Category")


transform!(SHLD, "Gender" => ByRow(x -> x == "Male" ? 0 : 1) => "Gender")

Row,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Heart Rate,Daily Steps,Sleep Disorder,BP_Systolic,BP_Diastolic
,Int64,Int64,Int64,String31,Float64,Int64,Int64,Int64,Int64,Int64,Int64,String15,Int64,Int64
1,1,0,27,Software Engineer,6.1,6,42,6,2,77,4200,None,126,83
2,2,0,28,Doctor,6.2,6,60,8,1,75,10000,None,125,80
3,3,0,28,Doctor,6.2,6,60,8,1,75,10000,None,125,80
4,4,0,28,Sales Representative,5.9,4,30,8,3,85,3000,Sleep Apnea,140,90
5,5,0,28,Sales Representative,5.9,4,30,8,3,85,3000,Sleep Apnea,140,90
6,6,0,28,Software Engineer,5.9,4,30,8,3,85,3000,Insomnia,140,90
7,7,0,29,Teacher,6.3,6,40,7,3,82,3500,Insomnia,140,90
8,8,0,29,Doctor,7.8,7,75,6,1,70,8000,None,120,80
9,9,0,29,Doctor,7.8,7,75,6,1,70,8000,None,120,80


In [47]:
using MLUtils

train, test = splitobs(SHLD; at = 0.70, shuffle = true)

(ObsView(::DataFrame, ::Vector{Int64})
 262 observations, ObsView(::DataFrame, ::Vector{Int64})
 112 observations)

In [48]:
train_idx = train.indices
test_idx  = test.indices

train = SHLD[train_idx, :]
test  = SHLD[test_idx, :]

Row,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Heart Rate,Daily Steps,Sleep Disorder,BP_Systolic,BP_Diastolic
,Int64,Int64,Int64,String31,Float64,Int64,Int64,Int64,Int64,Int64,Int64,String15,Int64,Int64
1,191,1,43,Teacher,6.7,7,45,4,2,65,6000,Insomnia,135,90
2,19,1,29,Nurse,6.5,5,40,7,1,80,4000,Insomnia,132,87
3,75,0,33,Doctor,6.0,6,30,8,1,72,5000,None,125,80
4,344,1,57,Nurse,8.1,9,75,3,2,68,7000,None,140,95
5,219,0,43,Engineer,7.8,8,90,5,1,70,8000,Sleep Apnea,130,85
6,29,0,30,Doctor,7.9,7,75,6,1,70,8000,None,120,80
7,346,1,57,Nurse,8.2,9,75,3,2,68,7000,Sleep Apnea,140,95
8,373,1,59,Nurse,8.1,9,75,3,2,68,7000,Sleep Apnea,140,95
9,118,1,37,Accountant,7.2,8,60,4,1,68,7000,None,115,75


In [49]:
features_select = train[:, [
    "Gender",    
    "Age", 
    "Sleep Duration",          
    "Quality of Sleep", 
    "Physical Activity Level", 
    "BMI Category", 
    "Heart Rate", 
    "Daily Steps", 
    "Stress Level", 
    "BP_Systolic",
    "BP_Diastolic"           
]]


features = Array(features_select)


labels = Array(train[!, "Sleep Disorder"])


features = float.(features)
labels = string.(labels)

262-element Vector{String15}:
 "Sleep Apnea"
 "None"
 "Sleep Apnea"
 "None"
 "None"
 "None"
 "None"
 "Insomnia"
 "None"
 "None"
 "None"
 "Sleep Apnea"
 "Insomnia"
 ⋮
 "None"
 "Insomnia"
 "None"
 "Insomnia"
 "Insomnia"
 "None"
 "None"
 "Sleep Apnea"
 "None"
 "None"
 "None"
 "None"

In [50]:
features_select_test = test[:, [
    "Gender",    
    "Age", 
    "Sleep Duration",          
    "Quality of Sleep", 
    "Physical Activity Level", 
    "BMI Category", 
    "Heart Rate", 
    "Daily Steps", 
    "Stress Level", 
    "BP_Systolic",
    "BP_Diastolic"           
]]


features_test = Array(features_select_test)
labels_test = Array(test[:, "Sleep Disorder"])
features_test = float.(features_test)
labels_test = string.(labels_test)

112-element Vector{String15}:
 "Insomnia"
 "Insomnia"
 "None"
 "None"
 "Sleep Apnea"
 "None"
 "Sleep Apnea"
 "Sleep Apnea"
 "None"
 "Insomnia"
 "Sleep Apnea"
 "None"
 "None"
 ⋮
 "None"
 "Sleep Apnea"
 "Insomnia"
 "None"
 "None"
 "Sleep Apnea"
 "Sleep Apnea"
 "Sleep Apnea"
 "None"
 "Sleep Apnea"
 "Insomnia"
 "None"

In [51]:
model = build_tree(labels, features)

Decision Tree
Leaves: 46
Depth:  13

In [52]:
print_tree(model, 5)

Feature 6 < 1.5 ?
├─ Feature 8 < 4550.0 ?
    ├─ Feature 8 < 4050.0 ?
        ├─ Sleep Apnea : 1/1
        └─ Sleep Apnea : 1/2
    └─ Feature 2 < 38.5 ?
        ├─ Feature 3 < 6.05 ?
            ├─ Feature 10 < 122.5 ?
                ├─ None : 2/3
                └─ 
            └─ Feature 2 < 30.5 ?
                ├─ None : 15/15
                └─ 
        └─ Feature 10 < 129.0 ?
            ├─ None : 31/31
            └─ Feature 2 < 42.5 ?
                ├─ 
                └─ 
└─ Feature 5 < 70.0 ?
    ├─ Feature 10 < 128.5 ?
        ├─ None : 8/8
        └─ Feature 2 < 42.5 ?
            ├─ Feature 1 < 0.5 ?
                ├─ 
                └─ Sleep Apnea : 4/4
            └─ Feature 2 < 43.5 ?
                ├─ 
                └─ 
    └─ Feature 10 < 130.5 ?
        ├─ None : 1/1
        └─ Feature 2 < 49.5 ?
            ├─ Feature 3 < 6.15 ?
                ├─ Sleep Apnea : 2/2
                └─ 
            └─ Feature 3 < 6.05 ?
                ├─ 
                └─ 

In [53]:
predictions = apply_tree(model, features_test)

112-element Vector{String15}:
 "Insomnia"
 "Sleep Apnea"
 "None"
 "Sleep Apnea"
 "None"
 "None"
 "Sleep Apnea"
 "Sleep Apnea"
 "None"
 "Insomnia"
 "Sleep Apnea"
 "None"
 "None"
 ⋮
 "None"
 "Sleep Apnea"
 "Sleep Apnea"
 "None"
 "None"
 "Sleep Apnea"
 "Sleep Apnea"
 "Sleep Apnea"
 "None"
 "Sleep Apnea"
 "Insomnia"
 "None"

In [54]:
unique(labels)

3-element Vector{String15}:
 "Sleep Apnea"
 "None"
 "Insomnia"

In [58]:
apply_tree_proba(model, [1, 43, 6.7, 7, 45, 4, 2, 65, 6000, 135, 90], ["Sleep Apnea", "None", "Insomnia"])

3-element Vector{Float64}:
 0.0
 0.0
 1.0

In [60]:
n_folds=5
accuracy = nfoldCV_tree(labels_test, features_test, n_folds)


Fold 1
Classes:  String15["Insomnia", "None", "Sleep Apnea"]
Matrix:   

3×3 Matrix{Int64}:
 2   1  0
 2  11  1
 1   0  4


Accuracy: 0.7727272727272727
Kappa:    0.6014492753623188

Fold 2
Classes:  String15["Insomnia", "None", "Sleep Apnea"]
Matrix:   

3×3 Matrix{Int64}:
 4   0  0
 0  14  0
 0   1  3


Accuracy: 0.9545454545454546
Kappa:    0.910569105691057

Fold 3
Classes:  String15["Insomnia", "None", "Sleep Apnea"]
Matrix:   

...

3×3 Matrix{Int64}:
 2   0  2
 0  11  0
 2   0  5


Accuracy: 0.8181818181818182
Kappa:    0.7046979865771812

Fold 5
Classes:  String15["Insomnia", "None", "Sleep Apnea"]
Matrix:   

3×3 Matrix{Int64}:
 4   1  0
 0  11  1
 0   1  4


Accuracy: 0.8636363636363636
Kappa:    0.7667844522968198

Mean Accuracy: 0.8636363636363636


5-element Vector{Float64}:
 0.7727272727272727
 0.9545454545454546
 0.9090909090909091
 0.8181818181818182
 0.8636363636363636

In [66]:
modelp = prune_tree(model, 0.8)

Decision Tree
Leaves: 22
Depth:  8

In [67]:
#burda hoca 0.6 almıştı. Bizim verimizde aşırı budama yapmamak için 0.8 aldık. 

In [68]:
print_tree(modelp, 3)

Feature 6 < 1.5 ?
├─ Feature 8 < 4550.0 ?
    ├─ Feature 8 < 4050.0 ?
        ├─ Sleep Apnea : 1/1
        └─ Sleep Apnea : 1/2
    └─ Feature 2 < 38.5 ?
        ├─ None : 86/91
        └─ 
└─ Feature 5 < 70.0 ?
    ├─ Feature 10 < 128.5 ?
        ├─ None : 8/8
        └─ 
    └─ Feature 10 < 130.5 ?
        ├─ None : 1/1
        └─ 


In [69]:
preds1=apply_tree(modelp,features_test)

n_folds=5
accuracy1 = nfoldCV_tree(preds1, features_test, n_folds)


Fold 1
Classes:  String15["Insomnia", "None", "Sleep Apnea"]
Matrix:   

3×3 Matrix{Int64}:
 3   0  0
 0  13  0
 0   0  6


Accuracy: 1.0
Kappa:    1.0

Fold 2
Classes:  String15["Insomnia", "None", "Sleep Apnea"]
Matrix:   

3×3 Matrix{Int64}:
 5   0  1
 0  11  0
 0   0  5


Accuracy: 0.9545454545454546
Kappa:    0.9273927392739274

Fold 3
Classes:  String15["Insomnia", "None", "Sleep Apnea"]
Matrix:   

...

3×3 Matrix{Int64}:
 2   0  0
 0  12  1
 0   1  6


Accuracy: 0.9090909090909091
Kappa:    0.83206106870229

Fold 5
Classes:  String15["Insomnia", "None", "Sleep Apnea"]
Matrix:   

3×3 Matrix{Int64}:
 4   0  0
 0  12  0
 0   0  6


Accuracy: 1.0
Kappa:    1.0

Mean Accuracy: 0.9727272727272727


5-element Vector{Float64}:
 1.0
 0.9545454545454546
 1.0
 0.9090909090909091
 1.0

In [72]:
#
#         FOLD1
#   1. Accuracy: 1.0 (Kusursuz Doğruluk)
#   Anlamı: Model, kendisine sorulan toplam 22 verinin (hastanın) tamamını doğru bilmiş.
#   Yorum: Hiçbir yanlış alarm (False Positive) veya gözden kaçırma (False Negative) yok. Doğruluk oranı %100.
#   2. Kappa: 1.0 (Mükemmel Uyum)
#   Anlamı: Kappa, modelin bu başarısının "şans eseri" olup olmadığını ölçer. Değerin 1.0 olması, modelin tahminleri ile gerçek veriler arasında tam bir uyuşma olduğunu gösterir.
#
#       FOLD2
#   1. Accuracy: 0.95 (22'de 21 Doğru)
#   Anlamı: Model, bu gruptaki toplam 22 hastadan 21 tanesinin durumunu doğru teşhis etmiş.
#   Yorum: Sadece tek bir hata (fire) var. Genel performans hala çok yüksek.
#   2. Kappa: 0.92 (Güvenilir Sonuç)
#   Anlamı: 0.81 - 1.00 arasındaki Kappa değeri literatürde "Neredeyse Mükemmel Uyum" olarak geçer.
#
#       FOLD3
#
#   Accuracy (1.0): 22 verinin 22'si de doğru tahmin edilmiş. Hata oranı %0.
#   Kappa (1.0): Modelin bu başarısı şans faktöründen tamamen arınmış durumda. Veri setindeki dengesizliğe (13 None'a karşı 3 Insomnia) rağmen model, azınlık sınıfları mükemmel ayırt etmiş.
#
#       FOLD4
#   1. Accuracy: 0.909 (%91 Başarı)
#   Anlamı: Model, bu gruptaki 22 hastadan 20 tanesini doğru bilmiş, 2 tanesinde hata yapmış.
#   Yorum: Ancak önceki %100'lük sonuçlardan sonra bu düşüş, veri setinde modelin zorlandığı bazı spesifik örnekler olduğunu gösterir.
#   2. Kappa: 0.83 (Güçlü Uyum)
#   Anlamı: Kappa değeri 0.83.
#   Yorum: 0.80 üzerindeki her değer literatürde "Mükemmel/Çok İyi Uyum" olarak kabul edilir. Yani modelin başarısı hala şansa bağlı değil. Ancak önceki Fold'larda 1.0 veya 0.92 görürken burada 0.83'e düşmesi, modelin bu veri grubundaki karmaşıklığı çözmekte biraz daha zorlandığını gösterir.
# 
#       FOLD5
#   Accuracy & Kappa: 1.0 (Kusursuz)
#   Bu gruptaki 22 hastanın (4 Insomnia, 12 None, 6 Sleep Apnea) tamamı doğru sınıflandırılmış.
#   Kappa'nın 1.0 olması uyumu gösterir

In [74]:
#   Önerilen modelin performansı, veri setindeki dengesizliklerin ve rastgeleliğin etkisini en aza indirmek amacıyla 5-Katmanlı Çapraz Doğrulama yöntemi
#   ile test edilmiştir.
#   Elde edilen sonuçlar incelendiğinde; modelin 5 farklı test katmanının 3'ünde %100 doğruluk oranına ulaştığı görülmüştür. En düşük performans 
#   gösterilen katmanda %90.90 başarı oranı korunmuştur. Tüm katmanların ortalaması alındığında, modelin Genel Doğruluk oranı %97.27 olarak çıkmıştır.
#   Bu sonuç, modelin 'Insomnia', 'Sleep Apnea' ve 'Sağlıklı' sınıflarını ayırt etmede yüksek bir kararlılığa sahip olduğunu göstermektedir.

In [71]:
n_subfeatures      = 2    # Burda değişken sayımdan dolayı 3 alabilirdik ama verisetimde çok baskın değişkenler olduğu için 2 aldık. Bu sayede çeşitliliği arttırmış olduk.
n_trees            = 20   # Kullandığım veri seti için  10 azdı, 20 yaptık.
partial_sampling   = 0.7
max_depth          = -1
min_samples_leaf   = 5
min_samples_split  = 10   # Leaf 5 ise split en az 10 olmalı ki bölünebilsin.
min_purity_increase= 0.1  # 2.0 çok fazlaydı, 0.1 yaptık. Notlarda 2.0 idi.

forest = build_forest(labels, features,
                      n_subfeatures,
                      n_trees,
                      partial_sampling,
                      max_depth,          
                      min_samples_leaf,
                      min_samples_split,
                      min_purity_increase;
                      rng=3)

Ensemble of Decision Trees
Trees:      20
Avg Leaves: 5.2
Avg Depth:  3.15

In [75]:
accuracy1=nfoldCV_forest(labels_test,features_test,5)


Fold 1
Classes:  String15["Insomnia", "None", "Sleep Apnea"]
Matrix:   

3×3 Matrix{Int64}:
 6   1  1
 0  10  1
 0   1  2


Accuracy: 0.8181818181818182
Kappa:    0.6986301369863015

Fold 2
Classes:  String15["Insomnia", "None", "Sleep Apnea"]
Matrix:   

3×3 Matrix{Int64}:
 5   1  1
 0  12  0
 0   0  3


Accuracy: 0.9090909090909091
Kappa:    0.8434163701067615

Fold 3
Classes:  String15["Insomnia", "None", "Sleep Apnea"]
Matrix:   

...

3×3 Matrix{Int64}:
 2   1  0
 1  11  0
 1   2  4


Accuracy: 0.7727272727272727
Kappa:    0.6014492753623188

Fold 5
Classes:  String15["Insomnia", "None", "Sleep Apnea"]
Matrix:   

3×3 Matrix{Int64}:
 1   0  0
 0  10  1
 1   0  9


Accuracy: 0.9090909090909091
Kappa:    0.838235294117647

Mean Accuracy: 0.8727272727272727


5-element Vector{Float64}:
 0.8181818181818182
 0.9090909090909091
 0.9545454545454546
 0.7727272727272727
 0.9090909090909091

In [77]:
#   Ortalama Doğruluk 0.8727 (%87.27) Model, genel olarak her 100 hastadan 87'sini doğru teşhis ediyor. Güçlü ve kabul edilebilir bir orandır.
#   (Tekli Karar Ağacı) %97 başarı varken, Random Forest ile %87'ye düşmesinin temel nedeni Veri Seti Boyutudur.
 
#   Küçük Veri Sorunu: Her Fold'da sadece 22 veri var. Tek bir Karar Ağacı, küçük veriyi kolayca "ezberleyebilir" (Overfitting) ve %100 sonuç verebilir.
#   Random Forest, veriyi ezberlemek yerine genel kuralları öğrenmeye çalışır. Bu %87.27'lik oran, modelinizin ezber yapmadığını, gerçek öğrenme 
#   kapasitesini yansıttığını gösterir. 
 
#   En Kötü Senaryo: Fold 4 (Accuracy: 0.77, Kappa: 0.60 Bu katman, modelin kafasının en çok karıştığı yer.
#   En İyi Senaryo: Fold 3 (Accuracy: 0.95) Burada model neredeyse kusursuz çalışmış. Demek ki veri setinin bazı bölümleri çok net ayrışabiliyor, 
#   bazıları (Fold 4'teki veriler) ise daha karmaşık (gürültülü).